[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flexfengfeng/dsai-m3-ml-genai/blob/main/L10-transformers-genai/notebooks/03_using_an_llm.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [1]:
# === Colab-compat setup (no-op when running locally) ===
# This whole block only DOES something on Google Colab. Locally, IN_COLAB is
# False, so the if-block is skipped entirely and only the threading caps run.
import os, sys

# Detect Colab by checking whether its special module has been imported.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab the repo isn't present, so we clone it, cd into the notebooks
    # folder, and install the libraries Colab doesn't ship with.
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
# setdefault means: only set the env var if it isn't already defined.
for _key, _val in [("OMP_NUM_THREADS", "1"), ("MKL_NUM_THREADS", "1"), ("TOKENIZERS_PARALLELISM", "false")]:
    os.environ.setdefault(_key, _val)

# L10 · NB 03 — Using a small LLM

> *We have the architecture. Now let's load a generative transformer and have it produce text. Same Lego bricks as NB 02 — but trained to predict the next token, with a causal mask and an instruction-tuning phase.*

Today's model: `SmolLM2-360M-Instruct` from Hugging Face. 361M parameters, ~720 MB on disk, runs on CPU in 1-3 seconds per response.

In this notebook we will:

1. Load the model and its tokenizer
2. Watch tokenisation happen end-to-end
3. Generate text — first with greedy decoding, then with sampling
4. Tweak temperature, top-k, top-p and see how they change output
5. Use chat templates the right way for instruction-tuned models

---

### Where you are in L10

Marcus asked for a shopping assistant. Sarah can't build it in one leap — she works through four notebooks, each answering the next question in the chain:

| Step | Notebook | Sarah's question |
|---|---|---|
| 1 | `01_monday_morning` | What can pretrained transformers already do, out of the box? |
| 2 | `02_attention_intuition` | *Why* do they work? What is this "attention" everyone talks about? |
| 3 | `03_using_an_llm` 👈 **you are here** | How do I drive a generative LLM myself — tokens, sampling, chat? |
| 4 | `04_rag_pipeline` | How do I ground it in NorthStar's catalogue? → the shopping assistant |

Right now you're on **step 3 of 4**. Each notebook stands on the one before it — run them in order.


## 1 · Setup


> **Why this cell first — and why it looks like a pile of switches.**
> Before Sarah touches a single model she runs this setup cell. It isn't glamorous, but every line earns its place:
>
> - `KMP_DUPLICATE_LIB_OK='TRUE'` — on macOS, PyTorch and the other scientific libraries each ship their own copy of the OpenMP maths runtime. When two copies load at once the kernel **crashes the instant you `import torch`**. This line tells them to coexist instead of fighting. (It's the same fix the repo's `.env` file applies — we repeat it here so the notebook also works in Colab and on a fresh machine.)
> - `torch.set_num_threads(1)` — pins the model to a single CPU thread. On a laptop this is *more* stable and predictable than letting it grab every core.
> - `HF_HUB_DISABLE_TELEMETRY` / `TRANSFORMERS_VERBOSITY='error'` / `warnings.filterwarnings('ignore')` — silence the download chatter and version warnings so the output you see is the output that *matters*.
>
> You'll see this same opening cell in every model notebook. Run it once at the top — it's the seatbelt before the drive.


In [2]:
# --- Environment guards (set BEFORE importing torch/transformers) ---
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home to Hugging Face
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # only show errors, hide info/warning chatter

import warnings
warnings.filterwarnings('ignore')  # mute version/deprecation warnings for cleaner output

# Core imports: time (timing), torch (the tensor engine), and the two Hugging Face
# "Auto" classes that pick the right tokenizer/model for any checkpoint name.
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(1)  # pin to 1 CPU thread — more stable/predictable on a laptop
torch.manual_seed(0)      # fix the random seed so runs are reproducible

# The model checkpoint we'll use: a small (361M-param) instruction-tuned LLM.
MODEL_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
# The tokenizer turns text <-> token IDs. It MUST match the model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# AutoModelForCausalLM = a "predict the next token" (causal/decoder) language model.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"Loaded in {time.time() - t0:.1f}s")
# numel() counts elements in each weight tensor; summing gives total parameters.
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Vocab size = how many distinct tokens the model knows (size of its output layer).
print(f"Vocab size: {tokenizer.vocab_size:,}")

Loading HuggingFaceTB/SmolLM2-360M-Instruct...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded in 18.2s
Parameters: 361,821,120
Vocab size: 49,152


**<span style="color:green">[Opus 4.8]</span> Reading these two printed numbers — and where every "MB" figure comes from:**

The cell just printed two numbers that are worth decoding, because they explain the download sizes quoted all through this lesson.

**`Parameters: 361,821,120`** — the total learnable weights, exactly the quantity the NB 02 §7 formula produces (`L · 12 · d_model² + vocab · d_model`). This is the "361M" in "SmolLM2-**360M**."

**`Vocab size: 49,152`** — the number of distinct tokens the tokenizer knows. It's *not* the same kind of number as the parameters: it's the count of rows in the embedding table **and** the length of the probability list the model outputs at every step (recall NB 03 §3 — one score per vocab token).

**How big is a model on disk? A one-line formula:**

```
file_size  ≈  n_parameters  ×  bytes_per_parameter
```

…where `bytes_per_parameter` depends on the numeric **precision** the weights are stored in:

| precision | bytes/param | note |
|---|---|---|
| fp32 (full) | 4 | the old default |
| fp16 / bf16 (half) | 2 | today's standard for shipping weights |
| int8 (quantised) | 1 | ~½ the size, tiny quality loss |
| 4-bit (quantised) | ~0.5 | the basis of QLoRA (see extensions) |

Plug it in and **every size figure in this lesson checks out**:

| model | params | × bytes | ≈ size | quoted as |
|---|---|---|---|---|
| distilbert-sst-2 (NB 01) | 67M | × 4 (fp32) | 268 MB | ~268 MB ✓ |
| bert-base-NER (NB 01) | 110M | × 4 | 440 MB | ~436 MB ✓ |
| bart-large-mnli (NB 01) | 400M | × 4 | 1.6 GB | ~1.6 GB ✓ |
| **SmolLM2-360M (here)** | **361.8M** | **× 2 (fp16)** | **~724 MB** | **~720 MB ✓** |

So SmolLM2's ~720 MB tells you it ships in **half precision** (361.8M × 2), while the older BERT-family models were quoted at full precision (× 4). Same formula, different precision.

*(A related rule of thumb: **RAM to run** a model ≈ its file size for the weights, plus some extra for activations and — during generation — the KV cache. That's why quantising to int8/4-bit is how large models are squeezed onto small GPUs.)*

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

## 2 · Tokenisation, step by step

Before the model sees any text, the tokenizer chops it into subword IDs. Let's see exactly what happens for one prompt.

**<span style="color:green">[Opus 4.8]</span> What tokenisation is, in plain English:**

A model can't read letters or words — it only does maths on **numbers**. So the very first thing that happens to your text is a translation step: chop it into small pieces and swap each piece for an ID number from a fixed dictionary (the **vocabulary**). Those pieces are called **tokens**, and they're usually **subwords** — bigger than a letter, often smaller than a word.

**Why subwords and not just whole words?** Two extremes, both bad:
- *One token per whole word* → the dictionary would need millions of entries and would still choke on any word it had never seen (names, typos, new slang).
- *One token per letter* → tiny dictionary, but sequences get enormously long and the model has to relearn spelling.

Subwords are the sweet spot. Common words get their own single token (`summer` → one piece); rare or complex words break into reusable chunks (`unbelievable` → `un` + `believ` + `able`). This way a **fixed ~50k-token vocabulary can spell essentially anything**, including words the model has never met, by assembling them from parts. The scheme most models use is **BPE** (byte-pair encoding), which builds the vocabulary by repeatedly merging the most common character pairs.

**Reading the output below:** the funny `Ġ` (or `Ċ`) you'll see marks a **leading space** — the tokenizer tracks spacing so it can perfectly rebuild the original text. Notice too that `'s` splits off, punctuation becomes its own token, and `£` fragments into pieces (rare symbols aren't worth a dedicated slot). Last thing worth internalising: **every model ships its own tokenizer**, so the same sentence becomes *different* numbers for SmolLM2 vs GPT‑2 vs Llama — which is exactly why you must always pair a model with its *own* tokenizer, never mix them.

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

In [3]:
text = "What's a good summer dress under £80?"

# Step 1: text -> token strings (for human readability)
# .tokenize() shows the subword PIECES the text is split into (Ġ marks a leading space).
token_strs = tokenizer.tokenize(text)
print(f"Input text: {text!r}")
print(f"Tokens    : {token_strs}")

# Step 2: tokens -> integer IDs (what the model actually consumes)
# .encode() maps each subword to its integer ID in the vocabulary. The model
# never sees letters — only these numbers.
token_ids = tokenizer.encode(text)
print(f"IDs       : {token_ids}")
print(f"Length    : {len(token_ids)} tokens")

Input text: "What's a good summer dress under £80?"
Tokens    : ['What', "'s", 'Ġa', 'Ġgood', 'Ġsummer', 'Ġdress', 'Ġunder', 'ĠÂ£', '8', '0', '?']
IDs       : [1780, 506, 253, 1123, 4018, 10272, 656, 10257, 40, 32, 47]
Length    : 11 tokens


Observations:
- Common words → one token each (`What`, `summer`)
- Possessives split (`'s`)
- Punctuation gets its own token
- Currency symbols like `£` may split into a few subword pieces depending on vocab

Every model has its OWN tokenizer. The same input string produces different token sequences for SmolLM2 vs MiniLM vs GPT-2 vs Llama. Don't mix tokenizers across models.

## 3 · The simplest generation — one forward pass

Before we wrap things up in `.generate()`, let's see what ONE forward pass looks like. Feed token IDs, get logits over the vocabulary for the NEXT token.

In [4]:
# Calling the tokenizer (vs .encode) returns a dict of tensors ready for the model.
# return_tensors='pt' -> PyTorch tensors; it also adds a batch dimension.
inputs = tokenizer(text, return_tensors='pt')
# input_ids shape is (batch, seq_len): one row per prompt, one column per token.
print(f"Input IDs shape: {inputs['input_ids'].shape}  (batch=1, seq_len)")

# One forward pass. no_grad() skips gradient tracking — we're predicting, not training.
with torch.no_grad():
    # **inputs unpacks input_ids AND attention_mask (1 = real token, 0 = padding).
    outputs = model(**inputs)

# logits = raw, unnormalised scores — one per vocab token, at EVERY position.
logits = outputs.logits   # shape: (batch, seq_len, vocab_size)
print(f"Logits shape   : {logits.shape}")

# For "what comes next", we only care about the prediction at the LAST position.
# [0, -1, :] = batch 0, last token, all vocab scores.
next_token_logits = logits[0, -1, :]
print(f"Next-token logits shape: {next_token_logits.shape}  (one number per vocab token)")

# topk finds the 5 highest-scoring candidate next tokens.
top5 = torch.topk(next_token_logits, 5)
print(f"\nTop 5 candidates for the next token:")
for prob_logit, tok_id in zip(top5.values, top5.indices):
    # softmax turns the raw logits into probabilities (0-1, summing to 1).
    prob = torch.softmax(next_token_logits, dim=-1)[tok_id]
    print(f"  {tokenizer.decode(tok_id):<20s}  logit={prob_logit:>6.2f}  prob={prob:.3f}")

Input IDs shape: torch.Size([1, 11])  (batch=1, seq_len)
Logits shape   : torch.Size([1, 11, 49152])
Next-token logits shape: torch.Size([49152])  (one number per vocab token)

Top 5 candidates for the next token:
  <|im_end|>            logit= 18.12  prob=0.660
  
                     logit= 16.38  prob=0.115
   I                    logit= 15.88  prob=0.069
   
                    logit= 14.38  prob=0.016
   �                    logit= 14.19  prob=0.013


That's the entire model: 360M parameters processed your input and produced a probability distribution over 49K vocabulary tokens. Pick one, append it, run again. That's text generation.

## 4 · The generation loop, by hand

`model.generate()` does this loop for us. Let's first do it manually to see exactly what's happening.

In [5]:
prompt = "The best way to learn programming is"
# Grab just the input_ids tensor (we'll keep appending to it as we generate).
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
print(f"Prompt: {prompt!r}")
print(f"Starting with {input_ids.shape[1]} tokens.")

# Greedy generation: pick the argmax token each step
max_new = 20
for step in range(max_new):
    with torch.no_grad():
        out = model(input_ids)  # forward pass on the whole sequence so far
    # argmax = take the single highest-probability next token (deterministic, "greedy").
    next_id = out.logits[0, -1, :].argmax().item()
    # append the chosen token to the running sequence, then loop and predict again
    input_ids = torch.cat([input_ids, torch.tensor([[next_id]])], dim=1)
    # stop on EOS — the model's "I'm done" end-of-sequence token
    if next_id == tokenizer.eos_token_id:
        break

# Decode IDs back to text; skip_special_tokens drops markers like <|im_end|>.
generated = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(f"\nGenerated:\n  {generated}")

Prompt: 'The best way to learn programming is'
Starting with 7 tokens.

Generated:
  The best way to learn programming is to start with a language that is easy to learn and has a large community of users. Python is


That's the entire generation algorithm. Every modern LLM — GPT-4, Claude, Llama 70B — does fundamentally this. The model is bigger and trained differently, but the loop is identical.

**<span style="color:green">[Opus 4.8]</span> Why the by-hand loop is slow — the KV cache:**

The loop above is correct but *quadratic*. Each step calls `model(input_ids)` on the **entire sequence so far**, re-computing keys and values for every earlier token even though those never change. Generating N tokens costs work proportional to 1 + 2 + … + N ≈ **O(N²)** — you redo almost all of the previous step's arithmetic every time.

`model.generate()` (next cell) avoids this with a **KV cache**: it stores each layer's key/value vectors for tokens already processed, so each new step computes Q/K/V for only the *one* new token and attends against the cached K/V. That's the main reason the library call is much faster — not a different algorithm, just no wasted recomputation. The trade-off is memory: the cache grows with sequence length, which is a real cost driver for long-context serving (the same O(N²) *memory* term from NB 02 §E2, now showing up at inference time).

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

## 5 · Use `model.generate()` for convenience

Doing the loop manually is illustrative but slow. The library's `.generate()` handles batching, caching, and stopping conditions efficiently.

In [6]:
prompt = "The best way to learn programming is"
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']

t0 = time.time()
# .generate() runs the same predict-append loop internally, but fast (with KV caching).
output_ids = model.generate(
    input_ids,
    max_new_tokens=40,      # cap on how many NEW tokens to produce
    do_sample=False,        # greedy — always take the argmax (no randomness)
    pad_token_id=tokenizer.eos_token_id,  # use EOS as padding to silence a warning
)
elapsed = time.time() - t0

generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Generated in {elapsed:.2f}s:\n  {generated}")

Generated in 6.42s:
  The best way to learn programming is to start with a language that is easy to learn and has a large community. Python is a great language to start with because it is simple, easy to learn, and has a large number of resources


## 6 · Sampling — temperature, top-k, top-p

Greedy is deterministic and produces boring text. Real applications use sampling. Let's vary the parameters and watch the output change for the same prompt.

In [7]:
# Instruction-tuned models expect chat-formatted prompts. Raw text often
# makes them emit EOS immediately and produce empty output — see Section 7
# of this notebook for the wrong-vs-right comparison. Use apply_chat_template.
messages = [{'role': 'user', 'content': 'Tell me a short bedtime story about a brave little robot.'}]
# apply_chat_template wraps the message in the special turn tokens the model was trained on.
chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(chat_prompt, return_tensors='pt')['input_ids']

# Three decoding settings, from deterministic (greedy) to high-randomness.
configs = [
    ('Greedy',              dict(do_sample=False)),                             # no randomness
    ('Temp=0.7, top-p=0.9', dict(do_sample=True, temperature=0.7, top_p=0.9)),  # balanced
    ('Temp=1.5, top-p=0.95',dict(do_sample=True, temperature=1.5, top_p=0.95)), # wild/creative
]

for name, cfg in configs:
    torch.manual_seed(42)  # consistent randomness across configs
    out = model.generate(
        # temperature: <1 sharpens the distribution (safer), >1 flattens it (riskier/creative).
        # top_p (nucleus): only sample from the smallest set of tokens whose probs sum to p.
        # do_sample=True turns on random sampling; False = greedy argmax.
        input_ids, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id, **cfg
    )
    # Slice off the prompt tokens so we print only the newly generated continuation.
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n=== {name} ===")
    print(text.strip()[:300])


=== Greedy ===
Once upon a time, in a faraway land, there was a tiny robot named Robby. Robby was a brave little robot who loved to explore the world. He had a special suit that made him invisible, so he could sneak around and do all sorts of exciting things.

One day

=== Temp=0.7, top-p=0.9 ===
Once upon a time, in a place called Robotville, there lived a tiny robot named Robby. Robby was a brave little robot who loved to play with his friends. One day, he decided to go on a mission to save the city from a very big and scary robot named Robo

=== Temp=1.5, top-p=0.95 ===
Livabo lived far away from here - it is up near North Carolina! Livabo didn't need to worry too much anymore though, for in these days people began sending robots to the moon. Now that made his family happy, so they looked forward to their trip every year. On June


**Notice how temperature controls creativity vs coherence.** Greedy is safest but repetitive. T=1.5 starts to drift but is more interesting. Production sweet spot: T=0.7-0.8 + top-p=0.9 for general use; T=0.2 for code; T=1.0 for creative writing.

Be aware: a 360M model has limits. Don't read too much into the *content* — focus on how each parameter shifts the distribution.

**<span style="color:green">[Opus 4.8]</span> The missing knob — top-k (the heading promised it):**

Section 6 is titled "temperature, top-**k**, top-p", but the demo only varies `temperature` and `top_p`. Here's the third knob so the picture is complete. All three act on the *same* next-token distribution, just cropping it differently before you sample:

- **temperature** — *reshapes* the whole distribution. `<1` sharpens (peaky, safe), `>1` flattens (more surprising). It changes probabilities but removes no tokens.
- **top-k** — *hard cap on count*. Keep only the **k highest-probability** tokens, renormalise, sample. `top_k=1` is exactly greedy. Simple, but k is fixed regardless of how confident the model is.
- **top-p (nucleus)** — *adaptive cap on mass*. Keep the smallest set of tokens whose probabilities sum to **p**, then sample. The cutoff *adapts*: few tokens when the model is confident, many when it's unsure — which is why top-p is usually preferred over top-k in practice.

You can combine them (`temperature` + `top_p`, or all three). Try adding `top_k=50` to one of the configs above and compare:

```python
out = model.generate(input_ids, max_new_tokens=60, do_sample=True,
                      temperature=0.7, top_k=50,
                      pad_token_id=tokenizer.eos_token_id)
```

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

## 7 · Chat templates — what instruction-tuned models actually expect

`SmolLM2-360M-Instruct` was fine-tuned on conversational data. It expects a SPECIFIC format with special tokens marking user/assistant turns. Get the format wrong and you'll get garbage.

**<span style="color:green">[Opus 4.8]</span> What a "chat template" is, in plain English:**

An instruction-tuned model like `SmolLM2-360M-Instruct` didn't learn from loose text — it learned from **conversations laid out in a very specific format**, with invisible marker tokens saying "the user's turn starts here," "the user's turn ends here," "now the assistant replies." During training it saw thousands of these, always in the same shape, so it only knows how to "be a helpful assistant" when the input arrives **in that exact shape**.

Think of it like a **form the model was trained to fill in**. Hand it the blank form laid out correctly and it knows precisely where to write its answer. Hand it a bare sentence with no form — like the "wrong way" cell just above — and it's confused: it often decides the input is already complete and just stops (emits the end token), giving you almost nothing.

**`apply_chat_template()` fills in that form for you.** You pass a tidy list of `{"role": ..., "content": ...}` messages, and it wraps them in the right marker tokens for *this particular model*. Run the next cell and read the printed string — you'll see SmolLM2's markers like `<|im_start|>user … <|im_end|>` and a trailing `<|im_start|>assistant` that says *"your turn now."* That final "your turn" cue is what `add_generation_prompt=True` adds.

The practical rule: **never hand-type these markers yourself.** Different models use different ones (`<|im_start|>`, `[INST]`, `<start_of_turn>`, …), and the template knows the correct set for whichever model you loaded. Build the `messages` list; let `apply_chat_template` do the formatting.

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

In [8]:
# Wrong way: just send the raw question with NO chat template.
wrong = "What is a CNN?"
input_ids = tokenizer(wrong, return_tensors='pt')['input_ids']
# The instruct model doesn't recognise this as a "user turn", so it often just
# stops immediately (emits EOS) and returns almost nothing.
out = model.generate(input_ids, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print('=== Without chat template (WRONG) ===')
print(tokenizer.decode(out[0], skip_special_tokens=True))

=== Without chat template (WRONG) ===
What is a CNN?


In [9]:
# Right way: apply the chat template
# messages is a list of role/content dicts — the standard chat format.
messages = [
    {"role": "user", "content": "What is a CNN?"}
]

# This adds the special <|im_start|> / <|im_end|> tokens and ends with assistant prompt
# add_generation_prompt=True appends the opening "assistant" marker so the model
# knows it's now its turn to reply.
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print('Formatted prompt:')
print(repr(formatted_prompt))  # inspect the exact string the model receives

input_ids = tokenizer(formatted_prompt, return_tensors='pt')['input_ids']
out = model.generate(input_ids, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
# Slice off the prompt so we keep only the model's answer.
response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
print('\n=== With chat template (RIGHT) ===')
print(response)

Formatted prompt:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is a CNN?<|im_end|>\n<|im_start|>assistant\n'

=== With chat template (RIGHT) ===
A Convolutional Neural Network (CNN) is a type of artificial neural network that is particularly well-suited for image and video processing tasks. It's a type of neural network that uses convolutional layers, pooling layers, and fully connected layers to extract features from input data, such as images or videos.

Convolutional layers are designed to process data at the level of individual pixels, allowing


See the difference. The chat-templated version produces a clean, focused answer. The raw version drifts.

**Always use `apply_chat_template`** for instruction-tuned models. Don't hand-roll prompts. The model's training defined exactly which special tokens delimit each turn — get it wrong and the model's behaviour is undefined.

## 8 · Multi-turn conversation

**<span style="color:green">[Opus 4.8]</span> The model has no memory — you carry it (plain English):**

This is the single most surprising thing about LLMs for newcomers: **the model remembers nothing between calls.** Each call to `generate()` is a blank slate — text goes in, text comes out, and the moment it's done the model forgets everything. It's a pure function, not a person taking notes.

So how does ChatGPT "remember" what you said three messages ago? **It doesn't — the app re-sends the entire conversation every single time.** That's exactly what the code below does: after the model replies, we `append` its answer *and* the next question to the `conversation` list, then send the **whole growing transcript** again. The model re-reads the full history from scratch on every turn and continues from there. The "memory" lives in *your list*, not in the model.

Two consequences that matter in practice:
- **Context has a size limit.** Since the whole history is re-sent each turn, a long conversation keeps growing until it hits the model's maximum context length. Past that, something has to be dropped or summarised — real chat apps do exactly this behind the scenes.
- **Longer history = slower and costlier.** Every turn re-processes all previous turns (recall the KV-cache / O(N²) note earlier), so late messages in a long chat cost more than early ones.

The upside of statelessness: it's simple and predictable. You have **total control** over what the model "knows" in any given call — it's precisely the messages you chose to include, nothing more, nothing hidden. RAG (NB 04) is built directly on this fact: if you want the model to know something, you *put it in the messages.*

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

In [10]:
# A multi-turn conversation: the list of messages IS the model's memory.
conversation = [
    {"role": "system", "content": "You are a helpful retail shopping assistant. Answer concisely."},  # sets behaviour/persona
    {"role": "user", "content": "What's a good fabric for a summer dress?"},
]

def chat(messages, max_new_tokens=120):
    # Format the full history, generate, and return only the new assistant text.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

response = chat(conversation)
print("Assistant:", response)

# Add it to the history and continue
# We append the model's reply AND the next user question, so turn 2 sees turn 1.
conversation.append({"role": "assistant", "content": response})
conversation.append({"role": "user", "content": "Is linen better than cotton for hot weather?"})

response = chat(conversation)  # now the model has the full context to answer the follow-up
print("\nAssistant:", response)

Assistant: A good fabric for a summer dress is cotton or linen.

Assistant: Yes, linen is better for hot weather.


**Notice three things:**

1. The **system prompt** sets behaviour ("be a retail assistant, be concise"). You can shape style and constraints here.
2. We **append** the assistant's reply to the conversation history before the next user turn. That's how the model "remembers" what it said.
3. The model gives reasonable, fluent retail-style answers — for a 360M-parameter model, that's remarkable. Imagine what 70B can do.

This is the loop that powers every chatbot, from a customer-service bot to ChatGPT. The model is bigger; the loop is the same.

## 9 · Recap

You now know:

- Tokenisation: text → subword IDs (model-specific)
- Forward pass: IDs → logits → next-token probability
- Generation: iteratively sample/argmax + append until done
- Sampling: temperature, top-k, top-p — tune the creativity dial
- Chat templates: format prompts the way the model expects
- Multi-turn: append history; LLMs are stateless, you supply the state

**Next:** an LLM by itself doesn't know YOUR data. To make it useful for product-specific Q&A, we need to give it the right context. That's RAG. NB 04.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · Streaming output

For interactive applications you want tokens to appear AS the model generates them — not in one big chunk at the end. `TextStreamer` handles this.

In [11]:
# TextStreamer prints tokens to the console AS they're generated, instead of
# waiting for the whole response — that's how chat UIs show text "typing out".
from transformers import TextStreamer

prompt = "Write a haiku about cosy autumn knitwear:"
messages = [{"role": "user", "content": prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(formatted, return_tensors='pt')['input_ids']

# skip_prompt=True: don't re-print the input; skip_special_tokens=True: hide markers.
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
# Pass streamer= to generate; the return value is ignored (_) since it's printed live.
_ = model.generate(input_ids, max_new_tokens=60, do_sample=True, temperature=0.8,
                   top_p=0.9, pad_token_id=tokenizer.eos_token_id, streamer=streamer)
print('\n[end of generation]')

Soft woolen sweater, snugly warm,
In the cozy winter season, snug and cool,
A perfect winter's gift, the fleece.

[end of generation]


## E2 · Stop sequences

You can tell `.generate()` to stop at a specific token sequence. Useful when you want to constrain output format (e.g., 'stop at a closing bracket').

In [12]:
# Stop generation when the model emits the string "STOP_HERE"
prompt_text = "List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.\n1."
input_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids']

# Find the token IDs for the stop string (might be multiple tokens)
# A stop "sequence" in text can map to several token IDs, so we match the whole list.
stop_ids = tokenizer.encode("STOP_HERE", add_special_tokens=False)
print(f"Stop token IDs: {stop_ids}")

# A simple per-step custom stop function via manual loop
out = input_ids.clone()
for _ in range(80):
    with torch.no_grad():
        nxt = model(out).logits[0, -1, :].argmax().item()  # greedy next token
    out = torch.cat([out, torch.tensor([[nxt]])], dim=1)    # append it
    # Check if the last few tokens match the stop sequence
    # Compare the tail of the output against stop_ids; if equal, we're done.
    if out[0][-len(stop_ids):].tolist() == stop_ids:
        break

print('Generated:')
print(tokenizer.decode(out[0], skip_special_tokens=True))

Stop token IDs: [48124, 79, 56, 14298]
Generated:
List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.
1. *Feta cheese*
2. *Tomato salad*
3. *Balsamic vinaigrette*
4. *Bran*
5. *Chicken*
6. *Eggs*
7. *Olive oil*
8. *Salt*
9. *Pepper*
10. *Bread*
11. *


Production frameworks (vLLM, llama.cpp) make stop sequences first-class. The mechanic is the same: watch the running output, break when you see what you wanted.